In [1]:
from pymilvus import MilvusClient

In [9]:
# client = MilvusClient("milvus_lab.db")

# client

In [26]:
# 方案1：用 MilvusClient 直接指定连接参数（推荐）
client = MilvusClient(
    uri="./milvus_lab.db",
    # 核心：关闭保活心跳，或调大心跳间隔
    kwargs={
        "grpc.keepalive_time_ms": 0,          # 关闭心跳发送
        "grpc.keepalive_timeout_ms": 10000,   # 超时时间（不影响）
        "grpc.keepalive_permit_without_calls": False,  # 无请求时不发心跳
        "grpc.http2.max_pings_without_data": 0,       # 禁止无数据时发ping
        "grpc.http2.min_time_between_pings_ms": 3600000,  # 心跳间隔设为1小时
        "grpc.http2.min_ping_interval_without_data_ms": 3600000  # 无数据时心跳间隔1小时
    }
)

# 类似DDL

In [18]:
if client.has_collection(collection_name="demo_collection"):
    client.drop_collection(collection_name="demo_collection")

client.create_collection(
    collection_name="demo_collection",
    dimension=768
)

In [4]:
from pymilvus import model

/opt/miniconda3/envs/milvus_lab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
# 检查版本。PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
import transformers
import pymilvus
print(f"transformers version: {transformers.__version__}")
print(f"pymilvus version: {pymilvus.__version__}")

transformers version: 5.1.0
pymilvus version: 2.6.9


In [ ]:
# 修复：降级 transformers 到兼容版本
# !pip install "transformers>=4.0,<5.0" --quiet

In [41]:
from IPython import embed


embedding_fn = model.DefaultEmbeddingFunction()
# Text strings to search from.
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]

vectors = embedding_fn.encode_documents(docs)
print("dims:",embedding_fn.dim,vectors[0].shape)

dims: 768 (768,)


In [12]:
len(vectors[0])

768

In [21]:
data = [
    {"id": i, "vector": vectors[i], "text": docs[i], "subject": "history"}
    for i in range(len(vectors))
]

print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))

Data has 3 entities, each with fields:  dict_keys(['id', 'vector', 'text', 'subject'])
Vector dim: 768


# insert  类似DML

In [23]:
res = client.insert(collection_name="demo_collection", data=data)
print("Insert result:", res)

Insert result: {'insert_count': 3, 'ids': [0, 1, 2]}


I0000 00:00:1771227785.892423 1328151 chttp2_transport.cc:1353] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0000 00:00:1771227785.892551 1328151 chttp2_transport.cc:1385] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms


I0000 00:00:1771227928.781100 1328444 chttp2_transport.cc:1353] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11, grpc_status:14}
E0000 00:00:1771227928.781214 1328444 chttp2_transport.cc:1385] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 40000ms


## search

In [24]:
query_vec = embedding_fn.encode_queries(["What is AI?"])

In [27]:
res = client.search(
    collection_name="demo_collection",  # target collection
    data=query_vec,  # query vectors
    limit=2,  # number of returned entities
    output_fields=["text", "subject"],  # specifies fields to be returned
)

print(res)

data: [[{'id': 1, 'distance': 0.40274232625961304, 'entity': {'text': 'Alan Turing was the first person to conduct substantial research in AI.', 'subject': 'history'}}, {'id': 0, 'distance': 0.23419252038002014, 'entity': {'text': 'Artificial intelligence was founded as an academic discipline in 1956.', 'subject': 'history'}}]]


I0000 00:00:1771229469.811460 1328444 chttp2_transport.cc:1353] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0000 00:00:1771229469.811539 1328444 chttp2_transport.cc:1385] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 160000ms


## filter search 用语意向量进行的搜索

In [31]:
docs = [
    "Machine learning has been used for drug design.",
    "Computational synthesis with AI algorithms predicts molecular properties.",
    "DDR1 is involved in cancers and fibrosis.",
]
vectors = embedding_fn.encode_documents(docs)
data = [
    {"id": 3 + i, "vector": vectors[i], "text": docs[i], "subject": "biology"
    ""}
    for i in range(len(vectors))
]

client.insert(collection_name="demo_collection", data=data)

res = client.search(
    collection_name="demo_collection",
    data=embedding_fn.encode_queries(["tell me AI related information"]),
    filter="subject == 'biology'",
    limit=1,
    output_fields=["text", "subject"],
)

print(res)


data: [[{'id': 4, 'distance': 0.2703055143356323, 'entity': {'text': 'Computational synthesis with AI algorithms predicts molecular properties.', 'subject': 'biology'}}]]


## 查询 query 按标量字段\id进行的精确查找

In [34]:
res = client.query(
    collection_name = 'demo_collection',
    filter='subject == "history"',
    output_fields=["text", "subject"]
)
print(res)

data: ["{'id': 0, 'text': 'Artificial intelligence was founded as an academic discipline in 1956.', 'subject': 'history'}", "{'id': 1, 'text': 'Alan Turing was the first person to conduct substantial research in AI.', 'subject': 'history'}", "{'id': 2, 'text': 'Born in Maida Vale, London, Turing was raised in southern England.', 'subject': 'history'}"], extra_info: {}


In [35]:
res = client.query(
    collection_name="demo_collection",
    ids=[0, 2],
    output_fields=["vector", "text", "subject"],
)


## delete

In [46]:
# res = client.delete(collection_name="demo_collection", ids=[2,3])
# print("Delete result:", res)
#再次运行返回为{}. 也可以
res = client.delete(collection_name="demo_collection", filter="id in [5,6]") #id是自增的 删了笑的就没了 除非手动插入
print("Delete result:", res)

Delete result: {}


I0000 00:00:1771234279.596558 1328388 chttp2_transport.cc:1353] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0000 00:00:1771234279.596896 1328388 chttp2_transport.cc:1385] unix:/var/folders/70/kzf41btn0bg3qdw5fd0fvjqr0000gn/T/tmpz39qza2n_milvus_lab.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 320000ms


In [ ]:
# 删除整个 collection
# client.drop_collection(collection_name="demo_collection")